In [1]:
import dromi.utils as du
import dromi.similarities as ds
import numpy as np
import os
import subprocess
import time

import plotly.express as px

# Python benchmark for simple cosine sim

In [2]:
n = 2000

array_a = np.array([x for x in range(n)])
array_b = np.array([x for x in range(n)])

matrix = np.array([[0 for _ in range(n)] for _ in range(n)])
start = time.time()
for i in range(n):
    for j in range(n):
        val = np.dot(array_a, array_b)/(np.linalg.norm(array_a)*np.linalg.norm(array_b))
        matrix[i][j] = val
time_python = time.time() - start

# OMP benchmark for 1 and 6 cores on Ryzen 5 laptop

In [18]:
os.chdir('../src/')
# os.system("export OMP_NUM_THREADS=1")
# time_omp_1 = float(subprocess.check_output(["./dromi_omp"]))
# os.system("export OMP_NUM_THREADS=6")
time_omp_6 = float(subprocess.check_output(["./dromi_omp"]))
os.chdir('../analysis/')

In [19]:
px.bar(x=['python', 'OMP 6 threads'], y=[time_python, time_omp_6])

In [11]:
seqs = ["AHPDYRMPIL"] * 5 + ["AHPDYRMPII"] * 5
max_len = len(seqs[0]) +2

In [12]:
padding_result = du.SequencePadding(
    seqs,
    max_len,
    method='ends',
    shuffle=False
    ).run()

sequences, sequences_padded = zip(*padding_result)

In [13]:
sequences_padded

(['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'L', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'L', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'L', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'L', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'L', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'I', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'I', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'I', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'I', '#', '#'],
 ['A', 'H', 'P', 'D', 'Y', 'R', 'M', 'P', 'I', 'I', '#', '#'])

In [14]:
blosum_array, blosum_dict, blosum_array_dict = du.create_blosum(
    21,
    "BLOSUM62",
    zero_characters=["#"],
    include_zero_characters=True
    )

aa_dict = du.aminoacid_names_dict(21, zero_characters=["#"])


In [15]:
sequences_int = np.vectorize(aa_dict.get)(sequences_padded)
sequences_blosum = np.vectorize(blosum_array_dict.get,signature='()->(n)')(sequences_int)
sequences_mask = sequences_int.astype(bool)

results = ds.calculate_similarities(
    sequences_blosum,
    max_len,
    sequences_mask,
    "./",
    batch_size=2,
    ksize=3,
    neighbours=1
    )


Generated 5 splits from 10 data points
 ------------  i: 0---------------------------- ------------  i: 0---------------------------- ------------  i: 0----------------------------
 ------------  i: 0---------------------------- ------------  i: 0----------------------------



###### j 0 ##########################
###### j 4 ################################ j 2 ################################ j 1 ##########################
###### j 3 ##########################
Time for finishing loop (i vs j) 0:00:00.014289


Time for finishing loop (i vs j) 0:00:00.015265 ------------  i: 1----------------------------Time for finishing loop (i vs j) 0:00:00.016458
Time for finishing loop (i vs j) 0:00:00.019250

 ------------  i: 1----------------------------###### j 0 ##########################

Time for finishing loop (i vs j) 0:00:00.024631 ------------  i: 1----------------------------

###### j 1 ##########################
 ------------  i: 1----------------------------
###### j 2 #############

In [18]:
results[2]

array([[1.    , 1.    , 1.    , 1.    , 1.    , 0.9937, 0.9937, 0.9937,
        0.9937, 0.9937],
       [1.    , 1.    , 1.    , 1.    , 1.    , 0.9937, 0.9937, 0.9937,
        0.9937, 0.9937],
       [1.    , 1.    , 1.    , 1.    , 1.    , 0.9937, 0.9937, 0.9937,
        0.9937, 0.9937],
       [1.    , 1.    , 1.    , 1.    , 1.    , 1.    , 0.9937, 0.9937,
        0.9937, 0.9937],
       [1.    , 1.    , 1.    , 1.    , 1.    , 0.9937, 0.9937, 0.9937,
        1.    , 0.9937],
       [0.9937, 0.9937, 0.9937, 1.    , 0.9937, 1.    , 1.    , 1.    ,
        1.    , 1.    ],
       [0.9937, 0.9937, 0.9937, 0.9937, 0.9937, 1.    , 1.    , 1.    ,
        1.    , 1.    ],
       [0.9937, 0.9937, 0.9937, 0.9937, 0.9937, 1.    , 1.    , 1.    ,
        1.    , 1.    ],
       [0.9937, 0.9937, 0.9937, 0.9937, 1.    , 1.    , 1.    , 1.    ,
        1.    , 1.    ],
       [0.9937, 0.9937, 0.9937, 0.9937, 0.9937, 1.    , 1.    , 1.    ,
        1.    , 1.    ]], dtype=float16)

In [37]:
sequences_blosum.shape

(10, 12, 21)

In [11]:
sequences_int.shape

(10, 10)